# Анализ финансовых транзакций и клиентов
## Тестовое задание 1

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Загрузка данных
transactions = pd.read_excel('../data/transactions_data.xlsx')
clients = pd.read_json('../data/clients_data.json')

print("Размер транзакций:", transactions.shape)
print("Размер клиентов:", clients.shape)

## 1. Очистка и подготовка данных

In [ ]:
# ---- Транзакции ----
# Удаление отрицательных сумм
transactions = transactions[transactions['amount'] > 0]

# Приведение дат
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], errors='coerce')
transactions = transactions.dropna(subset=['transaction_date'])

# ---- Клиенты ----
# Удаление записей без id
clients = clients.dropna(subset=['id'])

# Преобразование net_worth в число
clients['net_worth'] = pd.to_numeric(clients['net_worth'], errors='coerce')

# Заполнение пропусков возраста медианой
clients['age'] = clients['age'].fillna(clients['age'].median())

# Заполнение пола модой
clients['gender'] = clients['gender'].fillna(clients['gender'].mode()[0])

print("Пропуски после очистки:")
print("Транзакции:\n", transactions.isnull().sum())
print("Клиенты:\n", clients.isnull().sum())

## 2. Анализ данных

In [ ]:
# Топ-5 услуг по количеству заказов
top_services = transactions['service'].value_counts().head(5)
print("Топ-5 услуг по количеству заказов:\n", top_services)

In [ ]:
# Средняя сумма транзакций по городам
avg_by_city = transactions.groupby('city')['amount'].mean().sort_values(ascending=False)
print("Средняя сумма по городам (первые 5):\n", avg_by_city.head())

In [ ]:
# Услуга с наибольшей выручкой
revenue_by_service = transactions.groupby('service')['amount'].sum()
top_service = revenue_by_service.idxmax()
top_revenue = revenue_by_service.max()
print(f"Услуга с наибольшей выручкой: {top_service} ({top_revenue:,.2f})")

In [ ]:
# Процент транзакций по способам оплаты
payment_pct = transactions['payment_method'].value_counts(normalize=True) * 100
print("Доля способов оплаты:\n", payment_pct)

In [ ]:
# Выручка за последний месяц
last_date = transactions['transaction_date'].max()
last_month_start = last_date.replace(day=1)
last_month_revenue = transactions[transactions['transaction_date'] >= last_month_start]['amount'].sum()
print(f"Выручка за последний месяц (с {last_month_start.date()} по {last_date.date()}): {last_month_revenue:,.2f}")

## 3. Объединение данных и анализ по уровням активов

In [ ]:
# Переименование id
clients.rename(columns={'id': 'client_id'}, inplace=True)

# Объединение
merged = transactions.merge(clients, on='client_id', how='left')
print("Размер после объединения:", merged.shape)

In [ ]:
# Функция категоризации
def wealth_level(net_worth):
    if pd.isna(net_worth):
        return 'Неизвестно'
    elif net_worth < 100_000:
        return 'Низкий капитал (<100k)'
    elif net_worth <= 1_000_000:
        return 'Средний капитал (100k-1M)'
    else:
        return 'Высокий капитал (>1M)'

merged['wealth_level'] = merged['net_worth'].apply(wealth_level)

In [ ]:
# Выручка по уровням капитала
revenue_by_wealth = merged.groupby('wealth_level')['amount'].sum().sort_values(ascending=False)
print("Выручка по уровням капитала:\n", revenue_by_wealth)

## 4. Визуализация

In [ ]:
# Распределение сумм транзакций
plt.figure()
sns.histplot(transactions['amount'], bins=50, kde=True, color='skyblue')
plt.title('Распределение сумм транзакций')
plt.xlabel('Сумма')
plt.ylabel('Частота')
plt.savefig('../output/charts/distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Выручка по услугам (горизонтальная диаграмма)
plt.figure(figsize=(10,6))
revenue_by_service.sort_values().plot(kind='barh', color='coral')
plt.title('Выручка по услугам')
plt.xlabel('Выручка')
plt.ylabel('Услуга')
plt.tight_layout()
plt.savefig('../output/charts/revenue_by_service.png', dpi=150)
plt.show()

In [ ]:
# Зависимость средней суммы от возраста
age_avg = merged.groupby('age')['amount'].mean().reset_index()
plt.figure()
sns.scatterplot(data=age_avg, x='age', y='amount', alpha=0.7)
sns.regplot(data=age_avg, x='age', y='amount', scatter=False, color='red')
plt.title('Средняя сумма транзакции vs возраст')
plt.xlabel('Возраст')
plt.ylabel('Средняя сумма')
plt.savefig('../output/charts/avg_amount_vs_age.png', dpi=150)
plt.show()

## 5. Прогнозирование спроса на следующий месяц

In [ ]:
# Ежедневное количество транзакций
daily = transactions.groupby(transactions['transaction_date'].dt.date).size().reset_index()
daily.columns = ['date', 'count']
daily['day_num'] = (daily['date'] - daily['date'].min()).dt.days

# Модель
X = daily[['day_num']].values
y = daily['count'].values
model = LinearRegression()
model.fit(X, y)

# Прогноз на 30 дней
last_day = daily['day_num'].max()
future_days = np.arange(last_day+1, last_day+31).reshape(-1,1)
pred_counts = model.predict(future_days)
pred_total = pred_counts.sum()

print(f"Прогнозируемое количество транзакций в следующем месяце: {pred_total:.0f}")

# Сохранение прогноза
future_dates = pd.date_range(daily['date'].max() + pd.Timedelta(days=1), periods=30)
forecast_df = pd.DataFrame({'date': future_dates, 'predicted_count': pred_counts})
forecast_df.to_csv('../output/forecast.csv', index=False)

In [ ]:
# Визуализация прогноза
plt.figure()
plt.plot(daily['date'], daily['count'], 'o-', label='Факт')
plt.plot(future_dates, pred_counts, 'r--', label='Прогноз')
plt.title('Прогноз ежедневного спроса')
plt.xlabel('Дата')
plt.ylabel('Количество транзакций')
plt.legend()
plt.savefig('../output/charts/forecast.png', dpi=150)
plt.show()

## Выводы
- Данные очищены от аномалий и пропусков.
- Самая популярная услуга: ...
- Наибольшую выручку приносит услуга: ...
- Клиенты с высоким капиталом генерируют наибольшую выручку.
- Построены графики распределения, выручки, зависимости от возраста.
- Прогноз спроса на следующий месяц: ... транзакций.